# OpenSeesMP partitioned cantilever — 2 partitions

A minimal end-to-end example that builds a long brick block in
**apeGmsh**, partitions the mesh into 2 sub-domains with METIS, and
emits an **OpenSeesMP** Python script that runs under
`mpiexec -n 2 python model_mp.py`.

We use apeGmsh for **geometry, meshing, and the FEM broker** only.
The OpenSees model is written **by hand** with `openseespy` —
bypassing the `g.opensees.*` composite — so every parallel
concept is explicit:

* Each MPI rank owns one partition's elements.
* Boundary nodes between the two partitions are defined on **both**
  ranks — that is how OpenSeesMP shares DOFs across processes.
* Materials, time series, and the load pattern are declared on
  every rank; only local nodal loads / fixes are emitted per rank.
* Parallel solver stack: `Mumps` + `ParallelRCM`.

Geometry: a 20 × 2 × 2 cantilever fixed at `x=0`, with a downward
tip resultant on the `x=20` face.


In [1]:
from pathlib import Path
import json

from apeGmsh import apeGmsh

OUT = Path("generated/21_mppy_block")
OUT.mkdir(parents=True, exist_ok=True)


## 1. Geometry — a long block with named boundary faces

`add_box` creates the volume; the base / tip surfaces are picked
by bounding-box query through `g.model.queries.entities_in_bounding_box`
(a thin tolerance-aware wrapper over Gmsh's spatial picker).


In [2]:
Lx, Ly, Lz = 20.0, 2.0, 2.0
hx, hy, hz = 20, 2, 2     # element counts per direction (structured hex)
eps = 1.0e-6

g = apeGmsh(model_name="block_mp")
g.begin()

vol = g.model.geometry.add_box(0, 0, 0, Lx, Ly, Lz, label="Body")
g.model.sync()

base_dimtags = g.model.queries.entities_in_bounding_box(
    -eps, -eps, -eps,  eps, Ly + eps, Lz + eps, dim=2)
tip_dimtags = g.model.queries.entities_in_bounding_box(
    Lx - eps, -eps, -eps,  Lx + eps, Ly + eps, Lz + eps, dim=2)
base_tags = [t for _, t in base_dimtags]
tip_tags  = [t for _, t in tip_dimtags]

g.physical.add_volume([vol], name="Body")
g.physical.add_surface(base_tags, name="Base")
g.physical.add_surface(tip_tags,  name="Tip")
print("base faces:", base_tags, " tip faces:", tip_tags)


base faces: [1]  tip faces: [2]


## 2. Structured hex mesh

Transfinite + recombine on every curve / surface / volume gives a
clean hex8 mesh — predictable element count, easy to reason about
when checking partition boundaries.


In [3]:
# Walk all curves of the volume; pick edge node counts by direction.
curves_1d = [(d, t) for d, t in
             g.model.queries.boundary([(3, vol)],
                                       oriented=False, recursive=True)
             if d == 1]
for _, ctag in curves_1d:
    bb = g.model.queries.bounding_box(dim=1, tag=ctag)
    dx, dy, dz = bb[3] - bb[0], bb[4] - bb[1], bb[5] - bb[2]
    if dx > max(dy, dz):
        n = hx + 1
    elif dy > max(dx, dz):
        n = hy + 1
    else:
        n = hz + 1
    g.mesh.structured.set_transfinite_curve(ctag, n)

surfs_2d = [(d, t) for d, t in
            g.model.queries.boundary([(3, vol)], oriented=False, recursive=False)
            if d == 2]
for _, stag in surfs_2d:
    g.mesh.structured.set_transfinite_surface(stag)
    g.mesh.structured.set_recombine(stag, dim=2)

g.mesh.structured.set_transfinite_volume(vol)
g.mesh.generation.generate(3)
print(g.mesh.queries.get_fem_data(dim=3).inspect.summary())


99 nodes, 40 elements (hex8:40), bandwidth=94
  Physical groups (3):
    (2) "Base"                   9 nodes, 4 elems
    (2) "Tip"                    9 nodes, 4 elems
    (3) "Body"                   99 nodes, 40 elems
  Labels (1):
    (3) 'Body'                   99 nodes, 40 elems
  Element types (1):
    hex8         dim=3, order=1, npe=8, count=40


## 3. Renumber, then partition into 2 sub-domains

Renumber **before** partitioning so node IDs are contiguous from 1
— keeps the emitted OpenSees script clean.


In [4]:
g.mesh.partitioning.renumber(dim=3, method="rcm", base=1)
info = g.mesh.partitioning.partition(2)
print(info)
print(g.mesh.partitioning.summary())


PartitionInfo(2 parts: P1:112, P2:112)
Partitioning(model='block_mp'): 2 partition(s)
  points    : 12 partitioned entities
  curves    : 20 partitioned entities
  surfaces  : 11 partitioned entities
  volumes   : 2 partitioned entities


## 4. Pull the FEM broker and inspect partitions

`fem.elements.get(partition=p)` returns elements assigned to
partition `p`. `fem.nodes.get(partition=p)` returns every node
touched by those elements — including the shared interface layer
that will be defined on **both** ranks.


In [5]:
fem = g.mesh.queries.get_fem_data(dim=3)
print("partitions:", fem.partitions)
for p in fem.partitions:
    n_nodes = len(fem.nodes.get(partition=p))
    n_elems = sum(len(grp) for grp in fem.elements.get(partition=p))
    print(f"  P{p}: {n_nodes} nodes, {n_elems} elements")

ids0 = set(int(n) for n in fem.nodes.get(partition=fem.partitions[0]).ids)
ids1 = set(int(n) for n in fem.nodes.get(partition=fem.partitions[1]).ids)
shared = ids0 & ids1
print(f"shared interface nodes: {len(shared)}")


partitions: [1, 2]
  P1: 54 nodes, 20 elements
  P2: 54 nodes, 20 elements
shared interface nodes: 9


## 5. Per-partition supports and tip loads

`Base` and `Tip` are global PGs — intersect with each partition to
decide which `fix` / `load` lines that rank emits.


In [6]:
base_ids = set(int(n) for n in fem.nodes.get(pg="Base").ids)
tip_ids  = set(int(n) for n in fem.nodes.get(pg="Tip").ids)
print(f"global support nodes: {len(base_ids)}   global tip nodes: {len(tip_ids)}")

P_total = -1.0e6   # downward tip resultant [N]
load_per_node = P_total / len(tip_ids)


global support nodes: 9   global tip nodes: 9


## 6. Pack each partition into a small JSON payload

The OpenSeesMP script reads `model_p<pid>.json` at runtime — keeps
the script tiny and the data stable. `model_meta.json` carries the
global parameters (material, load magnitude, ndm/ndf).


In [7]:
def pack_partition(p: int) -> dict:
    nres = fem.nodes.get(partition=p)
    local_node_ids = [int(n) for n in nres.ids]
    local_node_set = set(local_node_ids)

    nodes = [
        [int(nid), float(x), float(y), float(z)]
        for nid, (x, y, z) in zip(nres.ids, nres.coords)
    ]

    elements = []
    for group in fem.elements.get(partition=p):
        if group.type_name != "hex8":
            continue
        for eid, conn in group:
            elements.append([int(eid), [int(n) for n in conn]])

    return {
        "pid_partition": p,
        "nodes": nodes,
        "elements": elements,
        "fixes":  sorted(local_node_set & base_ids),
        "loaded": sorted(local_node_set & tip_ids),
    }

partitions_sorted = fem.partitions
assert len(partitions_sorted) == 2, "this example expects exactly 2 partitions"

for rank, p in enumerate(partitions_sorted):
    payload = pack_partition(p)
    path = OUT / f"model_p{rank}.json"
    path.write_text(json.dumps(payload))
    print(f"wrote {path.name}: {len(payload['nodes'])} nodes, "
          f"{len(payload['elements'])} elems, "
          f"{len(payload['fixes'])} fixes, {len(payload['loaded'])} loads")

(OUT / "model_meta.json").write_text(json.dumps({
    "ndm": 3, "ndf": 3,
    "E": 30.0e9, "nu": 0.2, "rho": 2400.0,
    "load_per_node": load_per_node,
}))


wrote model_p0.json: 54 nodes, 20 elems, 9 fixes, 0 loads
wrote model_p1.json: 54 nodes, 20 elems, 0 fixes, 9 loads


104

## 7. Emit the OpenSeesMP script

Each rank loads its own `model_p<pid>.json`, defines its slice of
the model, then everyone runs a single static load step on the
parallel solver. `Mumps` is the standard MPI sparse direct solver
shipped with OpenSeesMP.


In [8]:
script = '''"""OpenSeesMP partitioned cantilever — run with:

    mpiexec -n 2 python model_mp.py
"""
import json
from pathlib import Path
import openseespy.opensees as ops

HERE = Path(__file__).parent
pid = ops.getPID()
nproc = ops.getNP()

if nproc != 2:
    raise RuntimeError(f"This script expects np=2 (got {nproc}). "
                       f"Run: mpiexec -n 2 python model_mp.py")

meta = json.loads((HERE / "model_meta.json").read_text())
data = json.loads((HERE / f"model_p{pid}.json").read_text())

ops.wipe()
ops.model("basic", "-ndm", meta["ndm"], "-ndf", meta["ndf"])

# Material — declared on every rank so element() can reference tag 1
ops.nDMaterial("ElasticIsotropic", 1, meta["E"], meta["nu"], meta["rho"])

# Local nodes (boundary nodes appear on both ranks — that is how
# OpenSeesMP knits the partitions together)
for nid, x, y, z in data["nodes"]:
    ops.node(nid, x, y, z)

# Local elements
for eid, conn in data["elements"]:
    ops.element("stdBrick", eid, *conn, 1)

# Local supports
for nid in data["fixes"]:
    ops.fix(nid, 1, 1, 1)

# Load pattern — every rank defines the pattern, only the rank that
# owns each tip node emits load()
ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)
for nid in data["loaded"]:
    ops.load(nid, 0.0, 0.0, meta["load_per_node"])

print(f"[pid={pid}] {len(data['nodes'])} nodes, "
      f"{len(data['elements'])} elems, "
      f"{len(data['fixes'])} fixes, {len(data['loaded'])} loads")

# Parallel analysis stack
ops.constraints("Transformation")
ops.numberer("ParallelRCM")
ops.system("Mumps")
ops.test("NormDispIncr", 1.0e-6, 50, 0)
ops.algorithm("Linear")
ops.integrator("LoadControl", 1.0)
ops.analysis("Static")
ok = ops.analyze(1)
if ok != 0:
    raise RuntimeError(f"[pid={pid}] analyze() failed: {ok}")

# Tip displacement — only the rank that owns tip nodes prints
for nid in data["loaded"]:
    uz = ops.nodeDisp(nid, 3)
    print(f"[pid={pid}] node {nid}  uz = {uz:.6e} m")

ops.wipe()
'''

(OUT / "model_mp.py").write_text(script)
print(f"wrote {OUT/'model_mp.py'}")


wrote generated\21_mppy_block\model_mp.py


## 8. Run the MP solve straight from the notebook

`mpiexec` is launched as a subprocess — `n=2` ranks, each running
the freshly-emitted `model_mp.py`. Requires:

* **MPI runtime** (MS-MPI on Windows, OpenMPI / MPICH on Linux/Mac)
* **OpenSeesMP-enabled `openseespy`** (the `openseespy-mp` wheel,
  or a custom build of OpenSees with MPI)

If either is missing the call is wrapped so the notebook still
renders cleanly — the equivalent shell command is printed instead.


In [9]:
import subprocess
import shutil
import sys

cmd = ["mpiexec", "-n", "2", sys.executable, "model_mp.py"]
print("$", " ".join(cmd), f"  (cwd={OUT})")

if shutil.which("mpiexec") is None:
    print("[skip] mpiexec not on PATH — install MS-MPI / OpenMPI to run.")
else:
    try:
        result = subprocess.run(
            cmd, cwd=str(OUT),
            capture_output=True, text=True, timeout=120,
        )
        print("--- stdout ---")
        print(result.stdout)
        if result.stderr:
            print("--- stderr ---")
            print(result.stderr)
        print(f"[exit code: {result.returncode}]")
    except subprocess.TimeoutExpired:
        print("[timeout] the MP run did not finish in 120 s")


$ mpiexec -n 2 C:\Users\nmora\venv\opensees_venv\Scripts\python.exe model_mp.py   (cwd=generated\21_mppy_block)
[skip] mpiexec not on PATH — install MS-MPI / OpenMPI to run.


Each rank prints its local node/element counts, then the rank that
owns the tip face prints `uz`. A monolithic `stdBrick` cantilever
of these dimensions under a 1 MN tip load gives roughly
`uz ≈ -P L^3 / (3 E I)` with `I = b h^3 / 12 ≈ 1.33` ⇒
`uz ≈ -7e-5 m` (order-of-magnitude check).


In [10]:
g.end()
